In [2]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

In [3]:
pois = gpd.read_file("C:/Users/Ā/Documents/python_class/dataset/delhi_pois.geojson")
wards = gpd.read_file("C:/Users/Ā/Documents/python_class/dataset/wards_delhi_c.geojson")


In [4]:
print("POIs:")
display(pois.head())

print("Wards:")
display(wards.head())

print("Number of POIs:", len(pois))
print("Number of wards:", len(wards))

print("POI CRS:", pois.crs)
print("Ward CRS:", wards.crs)

print("POI geometry type:")
print(pois.geometry.geom_type.value_counts())

print("Ward geometry type:")
print(wards.geometry.geom_type.value_counts())


POIs:


,sl_no,PoI_Type,Latitude,Longitude,geometry
0,1,Parks,28.469460,77.122980,POINT (77.12298 28.46946)
1,2,Schools,28.437668,77.141225,POINT (77.14122 28.43767)
2,3,Parks,28.438900,77.142180,POINT (77.14218 28.4389)
3,4,Playgrounds_Open_Source,28.447527,77.138000,POINT (77.138 28.44753)
4,5,amenity,28.452406,77.141672,POINT (77.14167 28.45241)


Wards:


,Ward_Name,Ward_No,geometry
0,DELHI CANTT CHARGE 1,CANT_1,"POLYGON ((77.13228 28.63154, 77.13644 28.62062..."
1,DELHI CANTT CHARGE 2,CANT_2,"POLYGON ((77.15429 28.62335, 77.15501 28.62228..."
2,DELHI CANTT CHARGE 4,CANT_4,"POLYGON ((77.15755 28.57578, 77.15672 28.57564..."
3,DELHI CANTT CHARGE 5,CANT_5,"POLYGON ((77.1348 28.57051, 77.13429 28.57048,..."
4,DELHI CANTT CHARGE 6,CANT_6,"POLYGON ((77.12157 28.59308, 77.12878 28.59029..."


Number of POIs: 50740
Number of wards: 290
POI CRS: EPSG:4326
Ward CRS: EPSG:4326
POI geometry type:
Point    50740
Name: count, dtype: int64
Ward geometry type:
Polygon    290
Name: count, dtype: int64


In [5]:
print("POI columns:")
print(pois.columns)

print("\nWard columns:")
print(wards.columns)

POI columns:
Index(['sl_no', 'PoI_Type', 'Latitude', 'Longitude', 'geometry'], dtype='str')

Ward columns:
Index(['Ward_Name', 'Ward_No', 'geometry'], dtype='str')


In [6]:
pois = pois.rename(columns={
    "sl_no": "id",
    "PoI_Type": "poi_type"
})

wards = wards.rename(columns={
   "Ward_No": "ward_id",
    "Ward_Name": "ward_name"
})

In [7]:
print("POI columns after renaming:")
print(pois.columns)

print("\nWard columns after renaming:")
print(wards.columns)


POI columns after renaming:
Index(['id', 'poi_type', 'Latitude', 'Longitude', 'geometry'], dtype='str')

Ward columns after renaming:
Index(['ward_name', 'ward_id', 'geometry'], dtype='str')


In [8]:
mapping = {

    # Education
    "Schools": "Education",
    "ITI_NIC_2018": "Education",
    "Library_Open_Source": "Education",

    # Health
    "Hospitals": "Health",
    "Pharmacetucal Labs": "Health",
    "Pharmacy": "Health",
    "Psychological_Counselling_Open_Source": "Health",
    "Veterinary Hospital": "Health",

    # Finance
    "ATM": "Finance",
    "Bank Branch": "Finance",
    "Bank Mitra": "Finance",
    "CSC_NIC_Through_API": "Finance",

    # Green space
    "Parks": "Green space",
    "Playgrounds_Open_Source": "Green space",
    "National_Parks_and_Bird_Sanctuaries_Open_Source": "Green space",

    # Commerce
    "shop": "Commerce",
    "Shopping_Malls_Open_Source": "Commerce",
    "Weekly_Markets_Open_Source": "Commerce",

    # Transport & Infrastructure
    "DMRC Stations": "Transport & Infrastructure",
    "Railway Junctions": "Transport & Infrastructure",
    "Level_Crossing_Open_Source": "Transport & Infrastructure",
    "Airports Helipads": "Transport & Infrastructure",
    "Aerodrome": "Transport & Infrastructure",
    "Transport_Nagar_Open_Source": "Transport & Infrastructure",
    "Parking_Open_Source": "Transport & Infrastructure",

    # Public Services & Utilities
    "Post Offices": "Public Services & Utilities",
    "office": "Public Services & Utilities",
    "Police_Station_Open_Source": "Public Services & Utilities",
    "Fire_Stations_Open_Source": "Public Services & Utilities",
    "Fuel_Stations_Open_Source": "Public Services & Utilities",
    "Toilets_Open_Source": "Public Services & Utilities",

    # Leisure, Tourism & Culture
    "Sports Facilities": "Leisure, Tourism & Culture",
    "Stadium": "Leisure, Tourism & Culture",
    "Golf_Course_Open_Source": "Leisure, Tourism & Culture",
    "Cinemas_Open_Source": "Leisure, Tourism & Culture",
    "tourism": "Leisure, Tourism & Culture",
    "Heritage & Tourism": "Leisure, Tourism & Culture",
    "historic": "Leisure, Tourism & Culture",
    "Museum_Tourism_Department": "Leisure, Tourism & Culture",
    "Lodging_Open_Source": "Leisure, Tourism & Culture"
}


In [9]:
# Create category column

pois["category"] = (
    pois["poi_type"]
    .map(mapping)
    .fillna("Other")
)


# Check categories

print("\nPOI categories:")
print(pois["category"].value_counts())



POI categories:
category
Finance                        16166
Other                          14030
Green space                     5750
Education                       5694
Commerce                        3405
Leisure, Tourism & Culture      2699
Public Services & Utilities     1443
Health                          1149
Transport & Infrastructure       404
Name: count, dtype: int64


In [10]:

# ------------------------------------------------------------
# 6. CHECK CRS
# ------------------------------------------------------------

print("\nPOI CRS:", pois.crs)
print("Ward CRS:", wards.crs)


# Make sure both layers use the same CRS

if pois.crs != wards.crs:
    pois = pois.to_crs(wards.crs)



POI CRS: EPSG:4326
Ward CRS: EPSG:4326


In [11]:

# ------------------------------------------------------------
# 7. SPATIAL JOIN
# ------------------------------------------------------------

# Attach ward information to every POI
# based on whether the POI lies within the ward polygon.

joined = gpd.sjoin(
    pois,
    wards[
        [
            "ward_id",
            "ward_name",
            "geometry"
        ]
    ],
    how="left",
    predicate="within"
)


In [12]:

# Remove automatically created index_right column

joined = joined.drop(
    columns="index_right"
)


# Check the joined data

print("\nJoined data:")
display(joined.head())




Joined data:


,id,poi_type,Latitude,Longitude,geometry,category,ward_id,ward_name
0,1,Parks,28.469460,77.122980,POINT (77.12298 28.46946),Green space,175,AYA NAGAR
1,2,Schools,28.437668,77.141225,POINT (77.14122 28.43767),Education,175,AYA NAGAR
2,3,Parks,28.438900,77.142180,POINT (77.14218 28.4389),Green space,175,AYA NAGAR
3,4,Playgrounds_Open_Source,28.447527,77.138000,POINT (77.138 28.44753),Green space,175,AYA NAGAR
4,5,amenity,28.452406,77.141672,POINT (77.14167 28.45241),Other,175,AYA NAGAR


In [13]:

# ------------------------------------------------------------
# 8. SELECT ONLY HEALTH POIs
# ------------------------------------------------------------

health = joined[
    joined["category"] == "Health"
].copy()


print(
    "\nNumber of Health POIs:",
    len(health)
)

display(health.head())


Number of Health POIs: 1149


,id,poi_type,Latitude,Longitude,geometry,category,ward_id,ward_name
21,22,Hospitals,28.476383,77.129023,POINT (77.12902 28.47638),Health,175,AYA NAGAR
96,97,Hospitals,28.463681,77.151760,POINT (77.15176 28.46368),Health,175,AYA NAGAR
124,125,Hospitals,28.468912,77.141100,POINT (77.1411 28.46891),Health,175,AYA NAGAR
169,170,Hospitals,28.495565,77.141839,POINT (77.14184 28.49556),Health,174,CHHATARPUR
225,226,Hospitals,28.513104,77.106843,POINT (77.10684 28.5131),Health,172,KISHANGARH


In [14]:

# ------------------------------------------------------------
# 9. COUNT HEALTH POIs IN EACH WARD
# ------------------------------------------------------------

health_counts = (
    health
    .dropna(subset=["ward_name"])
    .groupby(
        [
            "ward_id",
            "ward_name"
        ]
    )
    .size()
    .reset_index(
        name="num_health_pois"
    )
)


print("\nHealth POI counts by ward:")

display(health_counts)




Health POI counts by ward:


,ward_id,ward_name,num_health_pois
0,1,NARELA,5
1,10,TIMAR PUR,4
2,100,KARAM PURA,11
3,101,RAJA GARDEN,5
4,102,RAGHUBIR NAGAR,1
...,...,...,...
263,NDMC_5,NDMC CHARGE 5,8
264,NDMC_6,NDMC CHARGE 6,4
265,NDMC_7,NDMC CHARGE 7,9
266,NDMC_8,NDMC CHARGE 8,7


In [15]:
# ------------------------------------------------------------
# 10. FIVE WARDS WITH THE LARGEST NUMBER
#     OF HEALTH POIs
# ------------------------------------------------------------

top_5_health = (
    health_counts
    .sort_values(
        "num_health_pois",
        ascending=False
    )
    .head(5)
)


print(
    "\nTop 5 wards by number of Health POIs:"
)

display(top_5_health)



Top 5 wards by number of Health POIs:


,ward_id,ward_name,num_health_pois
91,192,GREATER KAILASH-I,18
262,NDMC_4,NDMC CHARGE 4,18
35,136,MATIALA,15
21,121,KUNWAR SINGH NAGAR,14
180,30,MUNDAKA,13


In [16]:

# ------------------------------------------------------------
# 11. MERGE HEALTH COUNTS BACK TO WARD POLYGONS
# ------------------------------------------------------------

health_ward_map = wards.merge(
    health_counts[
        [
            "ward_id",
            "num_health_pois"
        ]
    ],
    on="ward_id",
    how="left"
)


In [17]:

# ------------------------------------------------------------
# 12. REPLACE MISSING COUNTS WITH 0
# ------------------------------------------------------------

health_ward_map["num_health_pois"] = (
    health_ward_map["num_health_pois"]
    .fillna(0)
)



In [18]:

# ------------------------------------------------------------
# 13. FINAL RESULT
# ------------------------------------------------------------

print(
    "\nFinal ward-level Health POI data:"
)

display(
    health_ward_map[
        [
            "ward_name",
            "num_health_pois"
        ]
    ]
)


Final ward-level Health POI data:


,ward_name,num_health_pois
0,DELHI CANTT CHARGE 1,1.0
1,DELHI CANTT CHARGE 2,0.0
2,DELHI CANTT CHARGE 4,0.0
3,DELHI CANTT CHARGE 5,0.0
4,DELHI CANTT CHARGE 6,2.0
...,...,...
285,PREM NAGAR,0.0
286,SHASTRI NAGAR,2.0
287,VISHNU GARDEN,2.0
288,TUKHMIR PUR,0.0
